# 09_disk_scale: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/09_disk_scale/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Read bounded pages of scalar codes from disk.

Concepts: Read bounded pages of scalar codes from disk.
Run: uv run python modules/09_disk_scale/examples/01_from_scratch.py
Expected: Inspect the printed IDs or measurements and the checked invariant.
CPU: seconds on the included small workload. Uses local data; no Docker or model download.
"""

import tempfile
from pathlib import Path

from tinyvec.mmap import experiment


def main():
    with tempfile.TemporaryDirectory() as directory:
        result = experiment(Path(directory), rows=8193, dimensions=32, block=1024)
    print("code bytes", result["bytes_per_scan"], "recall", result["recall_at_10"])
    assert result["bytes_per_scan"] == 8193 * 32


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/09_disk_scale/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("faiss") is None:
    print("SKIP native comparison: --extra libs")
else:
    """Reopen a USearch graph through its memory-mapped view.

    Concepts: Reopen a USearch graph through its memory-mapped view.
    Run: uv run --extra libs python modules/09_disk_scale/examples/02_production.py
    Expected: Inspect the printed IDs or measurements and the checked invariant.
    CPU: seconds on the included small workload. Uses local data; no Docker or model download.
    """

    # course-extra: libs
    import tempfile
    from pathlib import Path

    import numpy as np
    from usearch.index import Index

    from common.index_lessons import unit_catalog

    def main():
        _, matrix, q = unit_catalog(320)
        with tempfile.TemporaryDirectory() as directory:
            path = str(Path(directory) / "products.usearch")
            index = Index(ndim=matrix.shape[1], metric="cos", dtype="f32")
            index.add(np.arange(len(matrix), dtype=np.uint64), matrix, threads=1)
            index.save(path)
            view = Index.restore(path, view=True)
            before = index.search(q, 5, threads=1).keys.tolist()
            after = view.search(q, 5, threads=1).keys.tolist()
            assert before == after
            print("restored IDs", after, "file bytes", Path(path).stat().st_size)
            del view, index

    if __name__ == "__main__":
        main()

## 03_measure

Source: `modules/09_disk_scale/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Measure a million generated product vectors with bounded allocation.

Concepts: Measure a million generated product vectors with bounded allocation.
Run: uv run python modules/09_disk_scale/examples/03_measure.py
Expected: Inspect the printed IDs or measurements and the checked invariant.
CPU: seconds on the included small workload. Uses local data; no Docker or model download.
"""

from scripts.measure_disk_scale import main as measure


def main():
    measure()


if __name__ == "__main__":
    main()

## 04_page_reads

Source: `modules/09_disk_scale/examples/04_page_reads.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Trace bounded disk graph reads and compare two fragment layouts.
Run: uv run python modules/09_disk_scale/examples/04_page_reads.py
CPU: seconds. Uses a temporary graph over included product vectors. No Docker or remote storage.
"""

import tempfile
from pathlib import Path

import numpy as np

from common.index_lessons import unit_catalog
from common.teaching import evidence
from tinyvec.hnsw import HNSW


def main():
    products, matrix, q = unit_catalog(240)
    graph = HNSW(matrix, M=6, efConstruction=24)
    # Keep compact sign codes in RAM, fixed-size adjacency records on disk.
    signs = np.packbits(matrix >= 0, axis=1)
    query_bits = np.packbits(q >= 0)
    links = np.full((len(matrix), 6), -1, dtype="<i4")
    for row, neighbors in graph.layers[0].items():
        links[row, : len(neighbors)] = neighbors
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "links.bin"
        links.tofile(path)
        seen = set()
        frontier = {graph.entry}
        with path.open("rb") as file:
            while frontier and len(seen) < 40:
                batch = sorted(
                    frontier, key=lambda i: int(np.bitwise_count(signs[i] ^ query_bits).sum())
                )[: min(4, 40 - len(seen))]
                for row in batch:
                    file.seek(row * links.strides[0])
                    neighbors = np.frombuffer(file.read(24), dtype="<i4")
                    seen.add(row)
                    frontier.update(int(n) for n in neighbors if n >= 0)
                frontier -= seen
        candidates = sorted(seen, key=lambda r: -float(matrix[r] @ q))[:10]
        exact = set(np.argsort(-(matrix @ q))[:10].tolist())
        # Estimate object bytes if each touched fragment had to be fetched whole.
        layouts = {}
        for per_fragment in [8, 64]:
            blocks = {row // per_fragment for row in seen}
            fetched = sum(min(per_fragment, len(matrix) - b * per_fragment) * 24 for b in blocks)
            layouts[str(per_fragment)] = {
                "fragments": len(blocks),
                "modeled_bytes": fetched,
                "useful_adjacency_bytes": len(seen) * 24,
            }
        result = {
            "products": len(products),
            "read_budget": 40,
            "actual_reads": len(seen),
            "fixed_record_bytes": 24,
            "recall_at_10": len(exact.intersection(candidates)) / 10,
            "final_ids": products.iloc[candidates].id.tolist(),
            "object_layout_estimates": layouts,
        }
    print(result)
    evidence("page-reads", result, tier=2)


if __name__ == "__main__":
    main()

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
import tempfile
from pathlib import Path

from tinyvec.mmap import experiment

with tempfile.TemporaryDirectory() as tmp:
    print(experiment(Path(tmp), 1024, 16, 128))

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.